# 11 – Ratings

Esplorazione e data cleaning del dataset `ratings.csv`.

> **Nota:** il file contiene ~124 milioni di righe. Il caricamento e le operazioni richiedono tempo e memoria significativi.

**Colonne:**
| Colonna | Descrizione |
|---|---|
| `username` | Nome utente MAL (FK → `profiles.csv`) |
| `anime_id` | ID dell'anime su MAL (FK → `details.csv`) |
| `status` | Stato di visione (`watching`, `completed`, `on_hold`, `dropped`, `plan_to_watch`) |
| `score` | Voto dell'utente (0–10; 0 = non votato) |
| `is_rewatching` | Indica se l'utente sta rivedendo l'anime (0/1/NaN) |
| `num_watched_episodes` | Numero di episodi guardati |

## 1. Import e caricamento dati

In [1]:
import pandas as pd
import numpy as np
from dataset_analyzer import analyze

In [2]:
df_rt = pd.read_csv('../datasets/ratings.csv')
print(f'Shape: {df_rt.shape}')
df_rt.info()
df_rt.head()

Shape: (124298357, 6)
<class 'pandas.DataFrame'>
RangeIndex: 124298357 entries, 0 to 124298356
Data columns (total 6 columns):
 #   Column                Dtype  
---  ------                -----  
 0   username              str    
 1   anime_id              int64  
 2   status                str    
 3   score                 int64  
 4   is_rewatching         float64
 5   num_watched_episodes  int64  
dtypes: float64(1), int64(3), str(2)
memory usage: 5.6 GB


,username,anime_id,status,score,is_rewatching,num_watched_episodes
0,--------788,30276,watching,7,0.0,3
1,--------788,28851,completed,7,0.0,1
2,--------788,41168,completed,7,0.0,1
3,--------788,22199,completed,10,0.0,24
4,--------788,16498,completed,10,0.0,25


In [3]:
n_originale = len(df_rt)

## 1.1 Rimozione righe senza `username`

Sono presenti 7 righe consecutive con `username` null (righe 100.051.758–100.051.764). Senza identificatore utente la riga non è collegabile a nessun profilo → rimozione.

In [4]:
print(f'Righe con username null: {df_rt["username"].isna().sum()}')
print()
print('Righe da rimuovere:')
print(df_rt[df_rt['username'].isna()])

df_rt.dropna(subset=['username'], inplace=True)
print(f'\nRighe dopo rimozione: {len(df_rt):,}')

Righe con username null: 7

Righe da rimuovere:
          username  anime_id         status  score  is_rewatching  \
100051758      NaN      1482       watching      9            0.0   
100051759      NaN      1735       watching      9            0.0   
100051760      NaN       121      completed     10            0.0   
100051761      NaN       136      completed      8            0.0   
100051762      NaN       269        on_hold      7            0.0   
100051763      NaN      1818        on_hold      7            0.0   
100051764      NaN      1535  plan_to_watch      6            0.0   

           num_watched_episodes  
100051758                    35  
100051759                    21  
100051760                    51  
100051761                    62  
100051762                    33  
100051763                    11  
100051764                     4  

Righe dopo rimozione: 124,298,350


## 1.2 Rimozione righe con `status = 'unknown'`

16.690 righe presentano `status = 'unknown'`: hanno sempre `score = 0` e `num_watched_episodes = 0`, quindi non contengono informazioni utili → rimozione.

In [5]:
mask_unknown = df_rt['status'] == 'unknown'
print(f'Righe con status=unknown : {mask_unknown.sum():,}')
print()
print('Caratteristiche di queste righe:')
print(df_rt[mask_unknown][['score', 'num_watched_episodes']].describe())

df_rt = df_rt[~mask_unknown].copy()
print(f'\nRighe dopo rimozione: {len(df_rt):,}')

Righe con status=unknown : 16,690

Caratteristiche di queste righe:
              score  num_watched_episodes
count  16690.000000          16690.000000
mean       0.306111              2.807969
std        1.561780             27.398930
min        0.000000              0.000000
25%        0.000000              0.000000
50%        0.000000              0.000000
75%        0.000000              1.000000
max       10.000000           1244.000000

Righe dopo rimozione: 124,281,660


## 2. Analisi colonna per colonna

### 2.1 `username`

In [6]:
analyze(df_rt['username'])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'username'  ══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    username
  dtype                          str
  Shape                          124,281,660
  Index range                    0 … 124298356
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     124,281,660
  Non-null values                124,281,660  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  309,312  (0.25%)
  Duplicate values               123,972,348

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length            

**Nessuna pulizia necessaria.**  
Nessun null (rimossi al passo 1.1). Non è una chiave primaria: uno stesso utente ha più righe (una per anime). Chiave esterna verso `profiles.csv`.

In [7]:
print(f'Null in username      : {df_rt["username"].isna().sum()}')
print(f'Utenti distinti       : {df_rt["username"].nunique():,}')

Null in username      : 0
Utenti distinti       : 309,312


### 2.2 `anime_id`

In [8]:
analyze(df_rt['anime_id'])


════════════════════════════════════════════════════════════════════════════════
═════════════════════  🔬  SERIES ANALYSIS  —  'anime_id'  ══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    anime_id
  dtype                          int64
  Shape                          124,281,660
  Index range                    0 … 124298356
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     124,281,660
  Non-null values                124,281,660  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Zeros                          0  (0.00%)
  Positives                      124,281,660   (100.00%)
  Negatives                      0   (0.00%)
  Unique values                  29,271  (0.02%)

 📐 Descriptive Statistics
──────────────────────────────────────────────

**Nessuna pulizia necessaria.**  
Nessun null, dtype già `int64`. Chiave esterna verso il dataset degli anime.

In [9]:
print(f'Null in anime_id    : {df_rt["anime_id"].isna().sum()}')
print(f'Anime distinti      : {df_rt["anime_id"].nunique():,}')

Null in anime_id    : 0
Anime distinti      : 29,271


### 2.3 `status`

In [10]:
analyze(df_rt['status'])


════════════════════════════════════════════════════════════════════════════════
══════════════════════  🔬  SERIES ANALYSIS  —  'status'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    status
  dtype                          str
  Shape                          124,281,660
  Index range                    0 … 124298356
  Kind                           STRING / OBJECT

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     124,281,660
  Non-null values                124,281,660  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Empty strings                  0
  Unique values                  5  (0.00%)
  Duplicate values               124,281,655

 📐 String Length Analysis
────────────────────────────────────────────────────────────────────────────────
  Min length                    

**Nessuna pulizia necessaria.**  
Nessun null, 5 valori distinti dopo la rimozione di `'unknown'` (passo 1.2). La distribuzione riflette il comportamento tipico degli utenti MAL: `completed` è il più frequente.

In [11]:
print(f'Null in status : {df_rt["status"].isna().sum()}')
print()
print('Distribuzione status:')
print(df_rt['status'].value_counts())

Null in status : 0

Distribuzione status:
status
completed        79138383
plan_to_watch    31631311
watching          5678165
dropped           4580651
on_hold           3253150
Name: count, dtype: int64


### 2.4 `score`

In [12]:
analyze(df_rt['score'])


════════════════════════════════════════════════════════════════════════════════
═══════════════════════  🔬  SERIES ANALYSIS  —  'score'  ═══════════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    score
  dtype                          int64
  Shape                          124,281,660
  Index range                    0 … 124298356
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     124,281,660
  Non-null values                124,281,660  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Zeros                          54,676,236  (43.99%)
  Positives                      69,605,424   (56.01%)
  Negatives                      0   (0.00%)
  Unique values                  11  (0.00%)

 📐 Descriptive Statistics
─────────────────────────────────────────────

**Nessuna pulizia necessaria.**  
Nessun null, dtype già `int64`, valori nell'intervallo 0–10. Il valore `0` indica che l'utente **non ha assegnato un voto** (~44% delle righe): è un comportamento atteso su MAL (un anime può essere in lista senza essere valutato).

In [13]:
print(f'Null in score : {df_rt["score"].isna().sum()}')
print(f'Valori distinti: {sorted(df_rt["score"].unique())}')
print()
n_unrated = (df_rt['score'] == 0).sum()
print(f'Righe con score=0 (non votato) : {n_unrated:,} ({n_unrated/len(df_rt)*100:.1f}%)')
print(f'Righe con score>0 (votato)     : {(df_rt["score"] > 0).sum():,} ({(df_rt["score"] > 0).mean()*100:.1f}%)')

Null in score : 0
Valori distinti: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10)]

Righe con score=0 (non votato) : 54,676,236 (44.0%)
Righe con score>0 (votato)     : 69,605,424 (56.0%)


### 2.5 `is_rewatching`

In [14]:
analyze(df_rt['is_rewatching'])


════════════════════════════════════════════════════════════════════════════════
═══════════════════  🔬  SERIES ANALYSIS  —  'is_rewatching'  ═══════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    is_rewatching
  dtype                          float64
  Shape                          124,281,660
  Index range                    0 … 124298356
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     124,281,660
  Non-null values                120,484,339  [█████████████████████████████░]  96.9%
  Null / NaN                     3,797,321  (3.06%)
  Zeros                          120,380,998  (96.86%)
  Positives                      103,341   (0.08%)
  Negatives                      0   (0.00%)
  Unique values                  2  (0.00%)

 📐 Descriptive Statistics
───────────────────────────────

**Pulizia necessaria:**  
- Dtype `float64` con valori `0.0`, `1.0` e `NaN` → conversione a booleano nullable (`boolean`)
- I ~3.797.321 null (~3%) sono strutturali: il campo non è compilato per tutti gli entry (assente per `plan_to_watch` e alcune versioni precedenti dell'API MAL)

In [15]:
print('Valori is_rewatching prima della conversione:')
print(df_rt['is_rewatching'].value_counts(dropna=False))

df_rt['is_rewatching'] = df_rt['is_rewatching'].astype('boolean')

print(f'\nis_rewatching dtype : {df_rt["is_rewatching"].dtype}')
print('Valori dopo la conversione:')
print(df_rt['is_rewatching'].value_counts(dropna=False))

Valori is_rewatching prima della conversione:
is_rewatching
0.0    120380998
NaN      3797321
1.0       103341
Name: count, dtype: int64

is_rewatching dtype : boolean
Valori dopo la conversione:
is_rewatching
False    120380998
<NA>       3797321
True        103341
Name: count, dtype: Int64


### 2.6 `num_watched_episodes`

In [16]:
analyze(df_rt['num_watched_episodes'])


════════════════════════════════════════════════════════════════════════════════
═══════════════  🔬  SERIES ANALYSIS  —  'num_watched_episodes'  ════════════════
════════════════════════════════════════════════════════════════════════════════

  Series name                    num_watched_episodes
  dtype                          int64
  Shape                          124,281,660
  Index range                    0 … 124298356
  Kind                           NUMERIC

 📊 Basic Counts
────────────────────────────────────────────────────────────────────────────────
  Total rows                     124,281,660
  Non-null values                124,281,660  [██████████████████████████████] 100.0%
  Null / NaN                     0  (0.00%)
  Zeros                          35,948,718  (28.93%)
  Positives                      88,332,942   (71.07%)
  Negatives                      0   (0.00%)
  Unique values                  1,919  (0.00%)

 📐 Descriptive Statistics
───────────────────────────

**Nessuna pulizia necessaria.**  
Nessun null, dtype già `int64`. Il valore `0` è atteso per entry con `status = 'plan_to_watch'` o per anime non ancora iniziati. Il valore massimo `65535` (= 2¹⁶ − 1) è un artefatto del tipo intero a 16 bit usato dall'API MAL per questo campo, ma riguarda un numero trascurabile di righe.

In [17]:
print(f'Null in num_watched_episodes : {df_rt["num_watched_episodes"].isna().sum()}')
print(f'Min  : {df_rt["num_watched_episodes"].min()}')
print(f'Max  : {df_rt["num_watched_episodes"].max()}')
print()
n_max = (df_rt['num_watched_episodes'] == 65535).sum()
print(f'Righe con valore 65535 (overflow): {n_max:,}')

Null in num_watched_episodes : 0
Min  : 0
Max  : 65535

Righe con valore 65535 (overflow): 2,691


### 2.7 Chiave composita `(username, anime_id)`

La combinazione `(username, anime_id)` identifica univocamente una riga: ogni utente ha al più un entry per anime. Verifichiamo su un campione di 2 milioni di righe.

In [18]:
sample_size = 2_000_000
df_sample = df_rt.sample(n=sample_size, random_state=42)
n_pk_dup = df_sample.duplicated(subset=['username', 'anime_id'], keep=False).sum()
print(f'Duplicati su (username, anime_id) nel campione di {sample_size:,} righe: {n_pk_dup}')

if n_pk_dup == 0:
    print('→ Chiave composita univoca nel campione, nessuna operazione richiesta.')
else:
    print('→ Presenza di duplicati, verificare sull\'intero dataset.')

Duplicati su (username, anime_id) nel campione di 2,000,000 righe: 0
→ Chiave composita univoca nel campione, nessuna operazione richiesta.


## 3. Data Cleaning

In [19]:
print('=== Riepilogo Dataset Pulito ===')
print(f'Righe originali      : {n_originale:>15,}')
print(f'Righe dopo cleaning  : {len(df_rt):>15,}')
print(f'Righe rimosse totali : {n_originale - len(df_rt):>15,}')
print()
print('Dtypes finali:')
print(df_rt.dtypes)
print()
print('Valori mancanti residui:')
print(df_rt.isnull().sum())

=== Riepilogo Dataset Pulito ===
Righe originali      :     124,298,357
Righe dopo cleaning  :     124,281,660
Righe rimosse totali :          16,697

Dtypes finali:
username                    str
anime_id                  int64
status                      str
score                     int64
is_rewatching           boolean
num_watched_episodes      int64
dtype: object

Valori mancanti residui:
username                      0
anime_id                      0
status                        0
score                         0
is_rewatching           3797321
num_watched_episodes          0
dtype: int64


In [20]:
df_rt.head()

,username,anime_id,status,score,is_rewatching,num_watched_episodes
0,--------788,30276,watching,7,False,3
1,--------788,28851,completed,7,False,1
2,--------788,41168,completed,7,False,1
3,--------788,22199,completed,10,False,24
4,--------788,16498,completed,10,False,25


## 4. Salvataggio dataset pulito

In [21]:
df_rt.to_csv('../datasets_cleaned/ratings_clean.csv', index=False)
print('Salvato: datasets_cleaned/ratings_clean.csv')

Salvato: datasets_cleaned/ratings_clean.csv
